In [1]:
import pandas as pd
from mlxtend.frequent_patterns import association_rules, apriori

# Read Dataset

In [2]:
sales_records = pd.read_csv('dataset/201904 sales reciepts.csv')
sales_records.head()

,transaction_id,transaction_date,transaction_time,sales_outlet_id,staff_id,customer_id,instore_yn,order,line_item_id,product_id,quantity,line_item_amount,unit_price,promo_item_yn
0,7,2019-04-01,12:04:43,3,12,558,N,1,1,52,1,2.50,2.50,N
1,11,2019-04-01,15:54:39,3,17,781,N,1,1,27,2,7.00,3.50,N
2,19,2019-04-01,14:34:59,3,17,788,Y,1,1,46,2,5.00,2.50,N
3,32,2019-04-01,16:06:04,3,12,683,N,1,1,23,2,5.00,2.50,N
4,33,2019-04-01,19:18:37,3,17,99,Y,1,1,34,1,2.45,2.45,N


In [3]:
product_sales = pd.read_csv('dataset/product.csv')
product_sales.head()

,product_id,product_group,product_category,product_type,product,product_description,unit_of_measure,current_wholesale_price,current_retail_price,tax_exempt_yn,promo_yn,new_product_yn
0,1,Whole Bean/Teas,Coffee beans,Organic Beans,Brazilian - Organic,It's like Carnival in a cup. Clean and smooth.,12 oz,14.40,$18.00,Y,N,N
1,2,Whole Bean/Teas,Coffee beans,House blend Beans,Our Old Time Diner Blend,Out packed blend of beans that is reminiscent ...,12 oz,14.40,$18.00,Y,N,N
2,3,Whole Bean/Teas,Coffee beans,Espresso Beans,Espresso Roast,Our house blend for a good espresso shot.,1 lb,11.80,$14.75,Y,N,N
3,4,Whole Bean/Teas,Coffee beans,Espresso Beans,Primo Espresso Roast,Our primium single source of hand roasted beans.,1 lb,16.36,$20.45,Y,N,N
4,5,Whole Bean/Teas,Coffee beans,Gourmet Beans,Columbian Medium Roast,A smooth cup of coffee any time of day.,1 lb,12.00,$15.00,Y,N,N


# Data Wrangling

## Merge Datasets

In [4]:
sales_records = sales_records[['transaction_id', 'transaction_date', 'product_id', 'quantity', 'customer_id', 'sales_outlet_id']]
product_sales = product_sales[['product_id', 'product_category','product']]

dataset = pd.merge(sales_records, product_sales, on='product_id', how='left')
dataset.head()

,transaction_id,transaction_date,product_id,quantity,customer_id,sales_outlet_id,product_category,product
0,7,2019-04-01,52,1,558,3,Tea,Traditional Blend Chai Rg
1,11,2019-04-01,27,2,781,3,Coffee,Brazilian Lg
2,19,2019-04-01,46,2,788,3,Tea,Serenity Green Tea Rg
3,32,2019-04-01,23,2,683,3,Coffee,Our Old Time Diner Blend Rg
4,33,2019-04-01,34,1,99,3,Coffee,Jamaican Coffee River Sm


## Remove Cup Sizes

In [5]:
dataset['product'] = dataset['product'].str.replace(' Rg', '')
dataset['product'] = dataset['product'].str.replace(' Sm', '')
dataset['product'] = dataset['product'].str.replace(' Lg', '')

In [6]:
dataset.head()

,transaction_id,transaction_date,product_id,quantity,customer_id,sales_outlet_id,product_category,product
0,7,2019-04-01,52,1,558,3,Tea,Traditional Blend Chai
1,11,2019-04-01,27,2,781,3,Coffee,Brazilian
2,19,2019-04-01,46,2,788,3,Tea,Serenity Green Tea
3,32,2019-04-01,23,2,683,3,Coffee,Our Old Time Diner Blend
4,33,2019-04-01,34,1,99,3,Coffee,Jamaican Coffee River


In [7]:
for product in dataset['product'].unique().tolist():
    print(f'-"{product}"')

-"Traditional Blend Chai"
-"Brazilian"
-"Serenity Green Tea"
-"Our Old Time Diner Blend"
-"Jamaican Coffee River"
-"Ethiopia"
-"English Breakfast"
-"Sustainably Grown Organic"
-"Earl Grey"
-"Cappuccino"
-"Espresso shot"
-"Latte"
-"Dark chocolate"
-"Columbian Medium Roast"
-"Oatmeal Scone"
-"Morning Sunrise Chai"
-"Peppermint"
-"Jumbo Savory Scone"
-"Lemon Grass"
-"Chocolate Chip Biscotti"
-"Spicy Eye Opener Chai"
-"Ginger Biscotti"
-"Chocolate Croissant"
-"Hazelnut Biscotti"
-"Cranberry Scone"
-"Scottish Cream Scone "
-"Croissant"
-"Almond Croissant"
-"Ginger Scone"
-"Ouro Brasileiro shot"
-"Organic Decaf Blend"
-"Chocolate syrup"
-"Hazelnut syrup"
-"Carmel syrup"
-"Sugar Free Vanilla syrup"
-"Jamacian Coffee River"
-"Guatemalan Sustainably Grown"
-"Civet Cat"
-"Chili Mayan"
-"Primo Espresso Roast"
-"Brazilian - Organic"
-"I Need My Bean! Diner mug"
-"Espresso Roast"
-"I Need My Bean! T-shirt"
-"I Need My Bean! Latte cup"


## Chose Product Subset

In [8]:
products_to_take = ['Cappuccino', 'Espresso shot', 'Latte', 'Dark chocolate', 'Chocolate Croissant', 'Ginger Scone', 'Croissant', 'Jumbo Savory Scone', 'Cranberry Scone', 'Hazelnut Biscotti', 'Almond Croissant', 'Oatmeal Scone', 'Chocolate Chip Biscotti', 'Ginger Biscotti']

In [9]:
print(products_to_take)
print(dataset['product'].nunique())
dataset = dataset[dataset['product'].isin(products_to_take)]


['Cappuccino', 'Espresso shot', 'Latte', 'Dark chocolate', 'Chocolate Croissant', 'Ginger Scone', 'Croissant', 'Jumbo Savory Scone', 'Cranberry Scone', 'Hazelnut Biscotti', 'Almond Croissant', 'Oatmeal Scone', 'Chocolate Chip Biscotti', 'Ginger Biscotti']
45


In [10]:
dataset[['product', 'product_category']].drop_duplicates().reset_index(drop=True) 

,product,product_category
0,Cappuccino,Coffee
1,Espresso shot,Coffee
2,Latte,Coffee
3,Dark chocolate,Drinking Chocolate
4,Oatmeal Scone,Bakery
5,Jumbo Savory Scone,Bakery
6,Chocolate Chip Biscotti,Bakery
7,Ginger Biscotti,Bakery
8,Chocolate Croissant,Bakery
9,Hazelnut Biscotti,Bakery


## Clean Transactions

In [11]:
dataset['transaction'] = dataset['transaction_id'].astype(str) + '_' + dataset['customer_id'].astype(str)
dataset.head()

,transaction_id,transaction_date,product_id,quantity,customer_id,sales_outlet_id,product_category,product,transaction
16,108,2019-04-01,40,1,65,3,Coffee,Cappuccino,108_65
17,112,2019-04-01,37,2,90,3,Coffee,Espresso shot,112_90
20,127,2019-04-01,41,2,116,3,Coffee,Cappuccino,127_116
21,134,2019-04-01,38,2,189,3,Coffee,Latte,134_189
22,135,2019-04-01,40,1,131,3,Coffee,Cappuccino,135_131


In [12]:
#Remove users who only purchase 1 item
num_items_per_transaction = dataset['transaction'].value_counts().reset_index()


In [13]:
valid_transaction = num_items_per_transaction[num_items_per_transaction['count'] > 1]['transaction'].tolist()

In [14]:
dataset = dataset[dataset['transaction'].isin(valid_transaction)]

## Popularity Recommendation Engine

In [24]:
product_recommendation = dataset.groupby(['product', 'product_category']).count().reset_index()

In [27]:
product_recommendation['product'].unique().tolist()

['Almond Croissant',
 'Cappuccino',
 'Chocolate Chip Biscotti',
 'Chocolate Croissant',
 'Cranberry Scone',
 'Croissant',
 'Dark chocolate',
 'Espresso shot',
 'Ginger Biscotti',
 'Ginger Scone',
 'Hazelnut Biscotti',
 'Jumbo Savory Scone',
 'Latte',
 'Oatmeal Scone']

In [28]:
product_recommendation = product_recommendation[['product', 'product_category', 'transaction_id']]
product_recommendation = product_recommendation.rename(columns={'transaction_id': 'num_of_transactions'})


In [29]:
product_recommendation['product'].unique().tolist()

['Almond Croissant',
 'Cappuccino',
 'Chocolate Chip Biscotti',
 'Chocolate Croissant',
 'Cranberry Scone',
 'Croissant',
 'Dark chocolate',
 'Espresso shot',
 'Ginger Biscotti',
 'Ginger Scone',
 'Hazelnut Biscotti',
 'Jumbo Savory Scone',
 'Latte',
 'Oatmeal Scone']

In [30]:
product_recommendation.to_csv('api/recommendation_objects/popularity_recommendation.csv', index = False)

# Apriori Recommendation Engine

In [15]:
dataset['product'] = dataset.apply(lambda row: f"{row['product']} ({row['product_category']})" if row['product'] == 'Dark chocolate' else row['product'], axis=1)
print(dataset['product'].str.len())

34       10
35       18
54        5
55       18
64       10
         ..
49880     5
49883    10
49884    15
49885    13
49886    19
Name: product, Length: 7241, dtype: int64


In [16]:
print(dataset['product'].unique().tolist())
print(dataset['product'].nunique())
print(dataset.columns)

['Cappuccino', 'Jumbo Savory Scone', 'Latte', 'Chocolate Chip Biscotti', 'Espresso shot', 'Hazelnut Biscotti', 'Chocolate Croissant', 'Dark chocolate (Drinking Chocolate)', 'Cranberry Scone', 'Croissant', 'Almond Croissant', 'Ginger Biscotti', 'Oatmeal Scone', 'Ginger Scone', 'Dark chocolate (Packaged Chocolate)']
15
Index(['transaction_id', 'transaction_date', 'product_id', 'quantity',
       'customer_id', 'sales_outlet_id', 'product_category', 'product',
       'transaction'],
      dtype='object')


In [17]:
train_basket = (dataset.groupby(['transaction', 'product', 'product_category'])['product'].count().reset_index(name = 'Count'))

In [18]:
train_basket.columns

Index(['transaction', 'product', 'product_category', 'Count'], dtype='object')

In [19]:
my_basket = train_basket.pivot_table(index = 'transaction', columns = 'product', values = 'Count')
my_basket = my_basket.fillna(0)

In [20]:
def encode_units(x):
    if x <= 0:
        return 0
    if x > 0:
        return 1
    
my_basket_sets = my_basket.applymap(encode_units)

In [21]:
my_basket_sets.head()

product,Almond Croissant,Cappuccino,Chocolate Chip Biscotti,Chocolate Croissant,Cranberry Scone,Croissant,Dark chocolate (Drinking Chocolate),Dark chocolate (Packaged Chocolate),Espresso shot,Ginger Biscotti,Ginger Scone,Hazelnut Biscotti,Jumbo Savory Scone,Latte,Oatmeal Scone
transaction,,,,,,,,,,,,,,,
1000_0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1
1002_0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0
1005_0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0
1006_0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0
1008_0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0


In [22]:
frequent_items = apriori(my_basket_sets, min_support=0.01, use_colnames=True)

c:\Users\Andrew\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:109: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [23]:
rules_basket = association_rules(frequent_items, metric="lift", min_threshold=1)
print(rules_basket)

                                antecedents  \
0                              (Cappuccino)   
1                        (Almond Croissant)   
2                 (Chocolate Chip Biscotti)   
3                        (Almond Croissant)   
4                         (Cranberry Scone)   
...                                     ...   
3671                   (Jumbo Savory Scone)   
3672                           (Cappuccino)   
3673                         (Ginger Scone)   
3674  (Dark chocolate (Drinking Chocolate))   
3675                                (Latte)   

                                            consequents  antecedent support  \
0                                    (Almond Croissant)            0.388889   
1                                          (Cappuccino)            0.157407   
2                                    (Almond Croissant)            0.153292   
3                             (Chocolate Chip Biscotti)            0.157407   
4                                    (Al

In [24]:
rules_basket[rules_basket['antecedents']== {'Latte'}].sort_values('confidence', ascending = False)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
108,(Latte),(Croissant),0.378086,0.155350,0.077675,0.205442,1.322449,0.018939,1.063044,0.392060
76,(Latte),(Chocolate Croissant),0.378086,0.184671,0.075103,0.198639,1.075641,0.005281,1.017431,0.113073
146,(Latte),(Ginger Scone),0.378086,0.181070,0.075103,0.198639,1.097032,0.006643,1.021925,0.142221
20,(Latte),(Almond Croissant),0.378086,0.157407,0.074074,0.195918,1.244658,0.014560,1.047894,0.316067
138,(Latte),(Ginger Biscotti),0.378086,0.144547,0.073560,0.194558,1.345980,0.018908,1.062091,0.413316
...,...,...,...,...,...,...,...,...,...,...
2518,(Latte),"(Ginger Biscotti, Croissant, Cappuccino)",0.378086,0.012346,0.010288,0.027211,2.204082,0.005620,1.015281,0.878412
3331,(Latte),"(Dark chocolate (Drinking Chocolate), Ginger B...",0.378086,0.016461,0.010288,0.027211,1.653061,0.004064,1.011051,0.635236
2814,(Latte),"(Oatmeal Scone, Espresso shot, Cappuccino)",0.378086,0.020062,0.010288,0.027211,1.356358,0.002703,1.007349,0.422457
3008,(Latte),"(Dark chocolate (Drinking Chocolate), Chocolat...",0.378086,0.017490,0.010288,0.027211,1.555822,0.003675,1.009993,0.574442


## Save in Json Format

In [25]:
product_categories = dataset[['product', 'product_category']].drop_duplicates().set_index('product').to_dict()['product_category']


In [26]:
recommendations_json = {

}
antecedents = rules_basket['antecedents'].unique()
for antecedent in antecedents:
    df_rec = rules_basket[rules_basket['antecedents'] == antecedent].sort_values('confidence', ascending = False)
    key = '_'.join(antecedent)
    recommendations_json[key] = []
    for _, row in df_rec.iterrows():
        rec_objects = row['consequents']
        for rec_object in rec_objects:
            already_exists = False
            for curr_rec_object in recommendations_json[key]:
                if rec_object == curr_rec_object['product']:
                    already_exists = True
            if already_exists:
                continue
            rec = {
                'product': rec_object,
                'product_category': product_categories[rec_object],
                'confidence': row['confidence'],
            }

            recommendations_json[key].append(rec)

In [27]:
import json
with open('api/recommendation_objects/apriori_recommendations.json', 'w') as f:
    json.dump(recommendations_json, f)